In [ ]:
# =====================================================================
# Dynamic-Room LGBM Validation
#   Part A: Cross-section sweeps (physics vs LGBM) at fixed room
#   Part B: HV/IGD Pareto topology metrics
# =====================================================================

import torch, numpy as np, math, time, json, glob, os
from scipy.stats import gaussian_kde
import matplotlib.pyplot as plt
import lightgbm as lgb
from pymoo.core.problem import ElementwiseProblem
from pymoo.algorithms.moo.nsga2 import NSGA2
from pymoo.algorithms.moo.age import AGEMOEA
from pymoo.operators.crossover.sbx import SBX
from pymoo.operators.mutation.pm import PM
from pymoo.operators.sampling.rnd import FloatRandomSampling
from pymoo.termination import get_termination
from pymoo.optimize import minimize
from pymoo.indicators.hv import HV
from pymoo.indicators.igd import IGD

try:
    device = torch.device('cuda'); torch.zeros(1, device=device)
except:
    device = torch.device('cpu')
print(f'Device: {device}')

# ============================================================
# 1. Load LGBM Model
# ============================================================
def find_file(pattern):
    for c in [pattern, f'/kaggle/working/{pattern}'] + sorted(glob.glob(f'/kaggle/input/**/{pattern}', recursive=True)):
        if os.path.exists(c): return c
    raise FileNotFoundError(pattern)

lgbm_m = lgb.Booster(model_file=find_file('lgbm_dynamic.txt'))
print('Model loaded.')

def enrich(xc,zc,Lx,Lz,rx,ry):
    xc_rel=xc/rx; Lx_rel=Lx/rx; area_ratio=(Lx*Lz)/(rx*3)
    dx=xc-10.0; dy=ry+100.0; dz_=zc+10.0; dist=np.sqrt(dx**2+dy**2+dz_**2)
    alpha_x=dx/dist; alpha_z=dz_/dist
    mL=xc-Lx/2; mR=rx-(xc+Lx/2); mB=zc-Lz/2; mT=3-(zc+Lz/2)
    aspect=Lx/(Lz+1e-6)
    return np.column_stack([xc,zc,Lx,Lz,np.full_like(xc,rx),np.full_like(xc,ry),
        xc_rel,Lx_rel,area_ratio,dist,alpha_x,alpha_z,mL,mR,mB,mT,aspect]).astype(np.float32)

def surr_predict(X_4d, rx, ry):
    return lgbm_m.predict(enrich(X_4d[:,0],X_4d[:,1],X_4d[:,2],X_4d[:,3],rx,ry))

# ============================================================
# 2. Physics Engine
# ============================================================
xBs,yBs,zBs=10.0,-100.0,-10.0; E=8; dB_s=0.075; lam=0.075
Lp=2;dref=abs(yBs)*1.5;Pd=40.0;Rth=0.2;Nd=-174.0;Bw=20e6;pm=0.2;nu=5;rz=3.0;STEP=0.05
PB=10**(Pd/10)*1e-3;N0f=10**(Nd/10)*1e-3*Bw;Z_HS=[1.5,1.625,1.75,1.875,2.0];N_Z=5

def gen_rwp(rx,ry,sim_time,dt=10):
    rng=np.random;rng.seed(777)
    rs=[0.0,rx,0.0,ry];hc=np.array([rx/2,ry/2]);hr=min(rx,ry)/3
    ts=int(sim_time/dt);p_t,p_s=0.6,0.3;tau_h,tau_n=1.5,0.1;v_h,v_n=0.2,1.0;g_h,g_n=0.6,0.2
    def gt():
        t=hc+(rng.rand(2)-0.5)*2*hr if rng.rand()<g_h else np.array([rs[0]+rng.rand()*(rs[1]-rs[0]),rs[2]+rng.rand()*(rs[3]-rs[2])])
        t[0]=np.clip(t[0],rs[0],rs[1]);t[1]=np.clip(t[1],rs[2],rs[3]);return t
    uh=1.5+0.5*rng.rand(nu);pos=np.zeros((nu,ts,2))
    up=np.zeros((nu,2));ur=['normal']*nu;us=[None]*nu;ut_=np.zeros((nu,2));ud=np.zeros((nu,2));usp=np.zeros(nu);uprem=np.zeros(nu)
    for i in range(nu):
        up[i]=[rs[0]+rng.rand()*(rs[1]-rs[0]),rs[2]+rng.rand()*(rs[3]-rs[2])]
        d2h=np.linalg.norm(up[i]-hc);ur[i]='hot' if d2h<=hr else 'normal'
        if rng.rand()<p_t:us[i]='transfer';ut_[i]=gt();dv=ut_[i]-up[i];ud[i]=dv/np.linalg.norm(dv);usp[i]=(v_h if ur[i]=='hot'else v_n)
        else:us[i]='pause';uprem[i]=rng.exponential(tau_h if ur[i]=='hot'else tau_n);pos[i,0,:]=up[i]
    for step in range(1,ts):
        for i in range(nu):
            if us[i]=='pause':
                uprem[i]-=dt;pos[i,step,:]=up[i]
                if uprem[i]<=0:us[i]='transfer';ut_[i]=gt();dv=ut_[i]-up[i];ud[i]=dv/np.linalg.norm(dv);usp[i]=(v_h if ur[i]=='hot'else v_n)
            else:
                md=usp[i]*dt;pd=ut_[i]-up[i]
                if np.linalg.norm(pd)<=md:
                    up[i]=ut_[i].copy();d2h=np.linalg.norm(up[i]-hc);ur[i]='hot' if d2h<=hr else 'normal'
                    if rng.rand()<p_s:us[i]='pause';uprem[i]=rng.exponential(tau_h if ur[i]=='hot'else tau_n)
                    else:ut_[i]=gt();dv=ut_[i]-up[i];ud[i]=dv/np.linalg.norm(dv);usp[i]=(v_h if ur[i]=='hot'else v_n)
                else:up[i]=up[i]+ud[i]*md;pos[i,step,:]=up[i]
    pts=np.zeros((nu*ts,3));idx=0
    for u in range(nu):
        for s in range(ts):pts[idx]=[pos[u,s,0],pos[u,s,1],uh[u]];idx+=1
    return pts

_PC={}
def build_room(rx,ry):
    nx,ny=int(rx/STEP),int(ry/STEP)
    xv=torch.linspace(0,rx,nx,device=device);yv=torch.linspace(0,ry,ny,device=device)
    Xg,Yg=torch.meshgrid(xv,yv,indexing='ij');xyf=torch.stack([Xg.flatten(),Yg.flatten()],dim=1)
    gl=[]
    for zh in Z_HS:gl.append(torch.stack([Xg.flatten(),Yg.flatten(),torch.full_like(Xg.flatten(),zh)],dim=1))
    gl=torch.cat(gl,dim=0);Ng=gl.shape[0]
    et=gen_rwp(rx,ry,100000,10);kde=gaussian_kde(et[:,:2].T,bw_method='scott')
    wxy=kde(xyf.cpu().numpy().T).flatten();wxy=wxy/wxy.sum()
    gw=torch.tensor(np.tile(wxy,N_Z),dtype=torch.float32,device=device);gw=gw/gw.sum()
    np.random.seed(42)
    ps=torch.tensor(2*np.pi*np.random.rand(Ng),dtype=torch.float32,device=device)
    tt=torch.tensor(-np.pi+2*np.pi*np.random.rand(Ng),dtype=torch.float32,device=device)
    eta=torch.tensor(10**((-15+5*np.random.rand(Ng))/10),dtype=torch.float32,device=device)
    na=torch.arange(E,dtype=torch.float32,device=device);dyB=torch.tensor(0.0-yBs,device=device)
    v1c=lam/(4*math.pi);v1pc=-(2*math.pi/lam);oE=1/math.sqrt(E)
    return gl,gw,ps,tt,eta,na,dyB,v1c,v1pc,oE

@torch.no_grad()
def phys_out(X_4d,rx,ry):
    key=(round(rx,2),round(ry,2))
    if key not in _PC:_PC[key]=build_room(rx,ry)
    gl,gw,ps,tt,eta,na,dyB,v1c,v1pc,oE=_PC[key]
    out=[]
    for i in range(0,len(X_4d),200):
        b=torch.tensor(X_4d[i:i+200],dtype=torch.float32,device=device);bo=torch.zeros(len(b),device=device)
        for j in range(len(b)):
            xc,zc,Lx,Lz=b[j,0],b[j,1],b[j,2],b[j,3]
            xg,yg,zg=gl[:,0],gl[:,1],gl[:,2];Ng=xg.shape[0]
            dx=xc-xBs;dy=dyB;dz=zc-zBs;Rw=torch.sqrt(dx**2+dy**2+dz**2);th=torch.atan2(dy,dx);ph=torch.acos(dz/Rw)
            kx=torch.sin(ph)*torch.cos(th);kz=torch.cos(ph)
            def rd(xs,zs):dd=xs-xBs;dz_=zs-zBs;L=torch.sqrt(dd**2+dy**2+dz_**2);return dd/L,dy/L,dz_/L
            x1,x2=xc-Lx/2,xc-Lx/2;x3,x4=xc+Lx/2,xc+Lx/2;z1,z3=zc-Lz/2,zc-Lz/2;z2,z4=zc+Lz/2,zc+Lz/2
            ux1,_,uz1=rd(x1,z1);ux2,_,uz2=rd(x2,z2);ux3,_,uz3=rd(x3,z3);ux4,_,uz4=rd(x4,z4)
            uma=torch.stack([ux1,ux2,ux3,ux4]);uzm=torch.stack([uz1,uz2,uz3,uz4])
            umin=uma.min(0).values;umax=uma.max(0).values;zmin=uzm.min(0).values;zmax=uzm.max(0).values
            dux=xg-xBs;duy=yg-yBs;duz=zg-zBs;Lu=torch.sqrt(dux**2+duy**2+duz**2);uux=dux/Lu;uuz=duz/Lu
            ix=torch.sigmoid(1000*(uux-umin))*torch.sigmoid(1000*(umax-uux))
            iz=torch.sigmoid(1000*(uuz-zmin))*torch.sigmoid(1000*(zmax-uuz));il=ix*iz
            d2x=xg-xc;d2y=yg;d2z=zg-zc;Ru=torch.sqrt(d2x**2+d2y**2+d2z**2);t1=d2x/Ru;t2=d2z/Ru
            ax=(1-il)*(kx-t1);az=(1-il)*(kz-t2)
            sx=torch.sinc((math.pi/lam)*Lx*ax);sz=torch.sinc((math.pi/lam)*Lz*az)
            p1=(2*math.pi/lam)*dB_s*na.unsqueeze(0)*torch.sin(th);a1=oE*torch.complex(torch.cos(p1),torch.sin(p1))
            v1m=v1c/Rw;v1p=v1pc*Rw;v1=torch.complex(v1m*torch.cos(v1p),v1m*torch.sin(v1p));H1=v1*a1.conj()
            p2=(2*math.pi/lam)*dB_s*na.unsqueeze(0)*torch.sin(tt).unsqueeze(1)
            a2=oE*torch.complex(torch.cos(p2),torch.sin(p2));v2m=eta*(lam/(4*math.pi*dref))
            v2=torch.complex(v2m*torch.cos(ps),v2m*torch.sin(ps));H2=v2.unsqueeze(1)*a2.conj()
            Ht=math.sqrt(E/Lp)*(H1+H2);fm=(Lx*Lz*sx*sz)/(lam*Ru)
            fp=(-(2*math.pi/lam)*(kx*xc+kz*zc)-(math.pi/2));fc=torch.complex(fm*torch.cos(fp),fm*torch.sin(fp))
            He=Ht*fc.unsqueeze(1);Hw=torch.sum(He,dim=1)/math.sqrt(E)
            dp=(torch.abs(Hw)**2)*pm*PB;it=(nu-1)*dp;sr=dp/(it+N0f)
            ab=math.pi/3.0;Ses=torch.zeros(Ng,device=device)
            rn=torch.sqrt((xg-xc)**2+yg**2+(zg-zc)**2+1e-12)
            for a in [0.0,math.pi/2,math.pi,3*math.pi/2]:
                dot=torch.abs((xg-xc)*math.cos(a)+yg*math.sin(a));cp=dot/rn;ph_b=torch.acos(torch.clamp(cp,0,1))
                Se=(math.pi-torch.clamp(2*ab-ph_b,min=0))/math.pi;Ses=Ses+torch.clamp(Se,1/3,1)
            sr_se=((Ses/4)*dp)/((Ses/4)*it+N0f);bits=(torch.log2(1+sr_se)<Rth).float();bo[j]=torch.sum(bits*gw)
        out.append(bo.cpu().numpy());torch.cuda.empty_cache()
    return np.concatenate(out)

print('Physics engine ready.')

# ============================================================
# Pick validation rooms
# ============================================================
# Test on a range of room sizes not explicitly in training grid
VAL_ROOMS = [(8.0,6.0), (12.0,10.0), (15.0,12.0), (18.0,14.0)]

for ROOM_X, ROOM_Y in VAL_ROOMS:
    print(f'\n{"="*60}')
    print(f'Room: {ROOM_X:.0f}×{ROOM_Y:.0f}m')
    print('='*60)

    # ============================================================
    # Part A: Cross-section sweep at a knee-like anchor
    # ============================================================
    # Quick coarse scan to find a feasible anchor
    np.random.seed(42)
    scan = np.column_stack([
        np.random.uniform(ROOM_X*0.3, ROOM_X*0.7, 50),
        np.random.uniform(0.5, 2.0, 50),
        np.random.uniform(1.0, ROOM_X*0.8, 50),
        np.random.uniform(0.1, 1.0, 50)])
    scan_o = phys_out(scan, ROOM_X, ROOM_Y)
    feas = scan_o < 0.10
    if feas.any():
        bi = np.argmin(scan[feas,2]*scan[feas,3])
        anchor = scan[feas][bi]
    else:
        anchor = np.array([ROOM_X/2, 1.5, ROOM_X*0.8, 0.25])
    print(f'Anchor: [{anchor[0]:.2f},{anchor[1]:.2f},{anchor[2]:.2f},{anchor[3]:.3f}]')

    Ns = 40
    sweeps = {
        'xc': (np.linspace(0.2, ROOM_X-0.2, Ns), [1,2,3]),
        'zc': (np.linspace(0.2, 2.8, Ns), [0,2,3]),
        'Lx': (np.linspace(0.05, ROOM_X-0.05, Ns), [0,1,3]),
        'Lz': (np.linspace(0.05, 2.95, Ns), [0,1,2]),
    }
    fig,axes=plt.subplots(2,2,figsize=(12,9))
    for (var,(vals,_)),ax in zip(sweeps.items(),axes.flat):
        vi={'xc':0,'zc':1,'Lx':2,'Lz':3}[var]
        Xs=np.tile(anchor,(Ns,1));Xs[:,vi]=vals
        phys=phys_out(Xs,ROOM_X,ROOM_Y);surr=surr_predict(Xs,ROOM_X,ROOM_Y)
        ax.plot(vals,phys*100,'b-',lw=1.5,label='Physics')
        ax.plot(vals,surr*100,'r--',lw=1.5,label='LGBM')
        ax.axvline(x=anchor[vi],color='k',ls=':',alpha=0.4);ax.axhline(y=10,color='gray',ls=':',alpha=0.4)
        ax.set_xlabel(f'{var}[m]');ax.set_ylabel('Outage[%]');ax.set_title(f'{var} sweep');ax.legend(fontsize=7);ax.grid(alpha=0.3)
    plt.suptitle(f'Part A: Cross-Section ({ROOM_X:.0f}×{ROOM_Y:.0f}m)',fontsize=13)
    plt.tight_layout();plt.savefig(f'validate_sweeps_{ROOM_X:.0f}x{ROOM_Y:.0f}.png',dpi=120);plt.show()

    # ============================================================
    # Part B: HV/IGD
    # ============================================================
    lb_p=np.array([0.0,0.0,0.05,0.05]);ub_p=np.array([ROOM_X,rz,ROOM_X,rz])
    class P(ElementwiseProblem):
        def __init__(self,phys_fn):
            self.pf=phys_fn;super().__init__(n_var=4,n_obj=2,n_ieq_constr=4,xl=lb_p,xu=ub_p)
        def _evaluate(self,x,out,*a,**k):
            xc,zc,Lx,Lz=x;out["G"]=[Lx/2-xc,xc+Lx/2-ROOM_X,Lz/2-zc,zc+Lz/2-rz]
            out["F"]=[Lx*Lz,float(self.pf(x.reshape(1,-1),ROOM_X,ROOM_Y)[0])]

    # True physics Pareto
    print('Running true physics AGE-MOEA (100 gen)...')
    algo_t=AGEMOEA(pop_size=200,n_offsprings=100,sampling=FloatRandomSampling(),
        crossover=SBX(0.9,15),mutation=PM(0.25,20))
    res_t=minimize(P(phys_out),algo_t,get_termination('n_gen',100),seed=42,verbose=False)
    Ft=res_t.F

    # LGBM surrogate Pareto
    print('Running LGBM surrogate NSGA-II (100 gen)...')
    algo_s=NSGA2(pop_size=200,n_offsprings=100,sampling=FloatRandomSampling(),
        crossover=SBX(0.9,15),mutation=PM(0.25,20),eliminate_duplicates=True)
    # Need physics-verified surrogate
    class SurrP(ElementwiseProblem):
        def __init__(self):super().__init__(n_var=4,n_obj=2,n_ieq_constr=4,xl=lb_p,xu=ub_p)
        def _evaluate(self,x,out,*a,**k):
            xc,zc,Lx,Lz=x;out["G"]=[Lx/2-xc,xc+Lx/2-ROOM_X,Lz/2-zc,zc+Lz/2-rz]
            out["F"]=[Lx*Lz,float(surr_predict(x.reshape(1,-1),ROOM_X,ROOM_Y)[0])]
    res_surr=minimize(SurrP(),algo_s,get_termination('n_gen',100),seed=42,verbose=False)
    # Physics-verify surrogate Pareto
    Fs=np.column_stack([res_surr.X[:,2]*res_surr.X[:,3],phys_out(res_surr.X,ROOM_X,ROOM_Y)])

    # Metrics
    nadir=np.max(np.vstack([Ft,Fs]),axis=0)*1.1;ideal=np.min(np.vstack([Ft,Fs]),axis=0)*0.9
    Ftn=(Ft-ideal)/(nadir-ideal);Fsn=(Fs-ideal)/(nadir-ideal)
    hv_t=HV(ref_point=np.array([1.1,1.1]))(Ftn)
    hv_s=HV(ref_point=np.array([1.1,1.1]))(Fsn)
    hv_dev=abs(hv_t-hv_s)/hv_t*100;igd_v=IGD(Ftn)(Fsn)
    print(f'  HV dev={hv_dev:.2f}%  IGD={igd_v:.4f}')

    fig2,ax2=plt.subplots(figsize=(8,5))
    ax2.scatter(Ft[:,1]*100,Ft[:,0],c='green',s=8,alpha=0.5,label=f'True Physics ({len(Ft)}pts)')
    ax2.scatter(Fs[:,1]*100,Fs[:,0],c='orange',s=8,alpha=0.5,label=f'LGBM Surrogate ({len(Fs)}pts)')
    ax2.axvline(x=10,color='red',ls='--',alpha=0.5)
    ax2.set_xlabel('Outage[%]');ax2.set_ylabel('Area[m²]')
    ax2.set_title(f'{ROOM_X:.0f}×{ROOM_Y:.0f}m: HV dev={hv_dev:.1f}% IGD={igd_v:.4f}')
    ax2.legend();ax2.grid(alpha=0.3)
    plt.tight_layout();plt.savefig(f'validate_pareto_{ROOM_X:.0f}x{ROOM_Y:.0f}.png',dpi=120);plt.show()

print('\nDone. All validation plots saved.')
from IPython.display import FileLink,display
for rx,ry in VAL_ROOMS:
    display(FileLink(f'validate_sweeps_{rx:.0f}x{ry:.0f}.png'))
    display(FileLink(f'validate_pareto_{rx:.0f}x{ry:.0f}.png'))